# DynaMate — Tutorial: Dynamic Multi-Agent Loop

This notebook walks through the complete **add → use → remove** lifecycle
of the DynaMate supervisor framework using a scientifically motivated example.

| Phase | Action |
|-------|--------|
| **1. Build** | Assemble the initial system and inspect the architecture |
| **2. Add** | Register a tool and create a dynamic agent via natural language |
| **3. Use** | Run a computation routed through the **Prompt Enhancer → Supervisor** pipeline |
| **4. Remove** | Delete the agent and tool, restoring the original architecture |

**Scientific context:** The Boltzmann thermal energy *kT* is the fundamental energy scale
in statistical mechanics and molecular simulation. We will register a `boltzmann_energy`
function as a tool, assign it to a dedicated `thermal_agent`, run a two-tool Arrhenius
calculation using the Prompt Enhancer, then cleanly remove both the agent and the tool.

---

### Framework overview

```
User prompt  (natural language — no tool/agent names required)
    │
    ▼
PromptEnhancer            ← lightweight LLM; reads live pool state and rewrites
    │                        the query with explicit routing hints
    ▼
Supervisor                ← LLM router; always reflects the current pool
    ├── tool_manager      ← registers tools, assigns them, adds/removes agents
    ├── shell_agent       ← runs shell commands (has terminal base tool)
    ├── compute_agent     ← general computation (starts empty)
    └── [dynamic agents]  ← created at runtime; persisted across sessions
              ▲
         AgentPool  (shared state)
         ├── _tool_registry   {name → StructuredTool}
         ├── _source_registry {name → source_code}  ← for persistence
         └── _agents          {name → {model, base_tools, extra_tools, ...}}
```

> **Package:** `dynamate` — built on LangGraph and `langgraph-supervisor`.
> Every pool mutation is written to `pool_state.json` immediately.

---
## 1 — Setup and System Initialization

DynaMate is assembled in **four ordered steps** to resolve the dependency between
the ToolManager (needs a pool reference) and the Supervisor (needs the ToolManager):

| Step | Call | Effect |
|------|------|--------|
| 1 | `PersistentAgentPoolWithSupervisor(...)` | Pool created; no supervisor yet |
| 2 | `pool.add_agent(...)` × 2 | `shell_agent` and `compute_agent` added |
| 3 | `build_tool_manager_v2(pool, model)` | ToolManager compiled, bound to the live pool |
| 4 | `pool.set_system_agents([tool_manager])` | **First supervisor build triggered** |

**Why `_is_dynamic=False` for the initial agents?**
Initial agents are always rebuilt from `main.py` on startup (they hold Python objects like
`ShellTool` that cannot be serialized). Setting `_is_dynamic=False` tells the persistent
pool to skip saving them to `pool_state.json`.

> **Isolation:** This tutorial writes to a temporary directory and does **not** affect
> your real `.dynamate/` session.

In [5]:
import sys, os, tempfile
sys.path.insert(0, os.path.abspath('..'))

import warnings
warnings.filterwarnings('ignore')

import dotenv
dotenv.load_dotenv()

from IPython.display import display, Image,Markdown
from langchain_openai import ChatOpenAI
from langchain_community.tools import ShellTool
from dynamate import (
    PersistentAgentPoolWithSupervisor,
    PersistentSaver,
    PoolStore,
    PromptEnhancer,
    build_tool_manager_v2,
    pretty_print_messages,
)

MODEL_NAME = 'gpt-4.1-mini'
model = ChatOpenAI(model=MODEL_NAME, temperature=0.0)

print(f'Model  : {MODEL_NAME}')
print('Imports: OK')

Model  : gpt-4.1-mini
Imports: OK


In [2]:
# ── Temporary state directory (isolated from real .dynamate/) ─────────────────
TMP = tempfile.mkdtemp(prefix='dynamate_tutorial_')

SUPERVISOR_PROMPT = '''You are the Supervisor managing a pool of agents.
- tool_manager : registers tools, assigns them to agents, and adds/removes agents.
- shell_agent  : runs shell commands and handles file-system tasks.
- compute_agent: performs calculations with its dynamically assigned tools.

Routing rules:
  * Add/register/assign/remove tools or agents -> tool_manager.
  * Shell or file-system tasks -> shell_agent.
  * Computation -> the agent that has the required tool.
  * If an agent for a task does not exist, ask tool_manager to create it first.
Assign work to one agent at a time.'''

# Step 1 — persistence layers
saver      = PersistentSaver(os.path.join(TMP, 'conversations'))
pool_store = PoolStore(os.path.join(TMP, 'pool_state.json'))

# Step 2 — create pool (no supervisor yet)
pool = PersistentAgentPoolWithSupervisor(
    supervisor_model=model,
    pool_store=pool_store,
    supervisor_prompt=SUPERVISOR_PROMPT,
    checkpointer=saver,
)

# Step 3 — add initial agents (_is_dynamic=False: not written to pool_state.json)
pool.add_agent(
    name='shell_agent',
    model=model,
    base_tools=[ShellTool()],
    system_prompt='You are a shell agent. Execute shell commands to answer requests.',
    _is_dynamic=False,
)
pool.add_agent(
    name='compute_agent',
    model=model,
    base_tools=[],
    system_prompt='You are a computation agent. Use your dynamically assigned tools for calculations.',
    _is_dynamic=False,
)

# Step 4 — build ToolManager and trigger first supervisor build
tool_manager = build_tool_manager_v2(pool, model)
pool.set_system_agents([tool_manager])

# Step 5 — build PromptEnhancer (wraps the same model; queries pool live on each call)
enhancer = PromptEnhancer(model=model, pool=pool)

print('System ready.')
print(f'Agents   : {pool.list_agents()}')
print(f'Tools    : {pool.list_registered_tools() or "(none)"}')
print(f'Enhancer : {enhancer.__class__.__name__} ready')
print(f'State    : {TMP}')

System ready.
Agents   : ['shell_agent', 'compute_agent']
Tools    : (none)
Enhancer : PromptEnhancer ready
State    : /tmp/735354.1.long/dynamate_tutorial_l12_nleo


---
## 2 — The Prompt Enhancer

Users don't know the names of the agents or tools in the pool. The **`PromptEnhancer`**
bridges this gap: it is a lightweight LLM layer that intercepts the raw user query,
inspects the current pool state, and rewrites the query with explicit routing hints
before passing it to the Supervisor.

```
User (no tool/agent names)
    │
    │  "What is the Arrhenius Boltzmann factor at 400 K for 0.30 eV?"
    ▼
PromptEnhancer.enhance()
    │  ┌─ reads live pool state (agents + their assigned tools)
    │  └─ one stateless chat-completion call
    │
    │  "...What is the Arrhenius Boltzmann factor at 400 K for 0.30 eV?
    │   Use thermal_agent — it should use both boltzmann_energy and
    │   arrhenius_factor tools to compute the answer step by step."
    ▼
Supervisor  →  thermal_agent  →  boltzmann_energy + arrhenius_factor
```

### How it works

| Property | Detail |
|----------|--------|
| **Context** | On every call, `_build_pool_context()` queries the live pool via `list_agents()` and `list_agent_tools()` — always reflects current state |
| **LLM call** | Single `model.invoke([SystemMessage, HumanMessage])` — no ReAct loop, no graph |
| **Output** | Original question kept verbatim + routing instruction appended |
| **Fallback** | If no agents have tools yet, the original input is returned unchanged |

> **Important:** The Prompt Enhancer is instantiated once (Step 5 of `build_system`)
> but its context is **always fresh** because it queries the pool at call time.
> Adding a new agent or assigning a new tool is immediately visible to the enhancer.

---
## 2 — Initial Architecture

The default pool contains **three agents** connected through the Supervisor:

```
Supervisor
├── tool_manager   ← manages the pool (register / assign / add / remove)
├── shell_agent    ← runs shell commands  [base tool: terminal]
└── compute_agent  ← no tools yet; ready to receive them dynamically
```

The graph below is generated from the **live compiled LangGraph supervisor**.
It always reflects the current state of the pool — run this cell again after
adding or removing agents to see the updated graph.

**Rebuild cost summary:**

| Operation | What is rebuilt |
|-----------|----------------|
| `assign_tool(tool, agent)` | Only that agent |
| `add_agent(name, ...)` | That agent + the supervisor |
| `remove_tool(name)` | All agents that had it + their assignments |
| `remove_agent(name)` | The supervisor only |

> **`compute_agent` is intentionally empty.** It cannot perform any calculation
> until at least one tool is assigned — this is by design.

In [3]:
def print_pool_state(pool, label='POOL STATE'):
    'Print a concise snapshot of the current pool agents and their tools.'
    sep = '=' * 58
    print(f'\n{sep}')
    print(f'  {label}')
    print(sep)
    print(f'  Agents   : {pool.list_agents()}')
    print(f'  Registry : {pool.list_registered_tools() or "(empty)"}')
    print()
    for name in pool.list_agents():
        entry = pool._agents[name]
        base  = [t.name for t in entry['base_tools']]
        extra = [t.name for t in entry['extra_tools']]
        print(f'  [{name}]')
        print(f'    base tools    : {base or "(none)"}')
        print(f'    assigned tools: {extra or "(none)"}')
    print(sep)

In [6]:
print('── Initial Architecture ─────────────────────────────────────')
#display(Image(pool.supervisor.get_graph().draw_mermaid_png()))

mermaid_code = pool.supervisor.get_graph().draw_mermaid()
# Add a custom style for a node named 'supervisor'
mermaid_code += "\nclassDef supervisor fill:#f96,stroke:#333,stroke-width:4px;"
mermaid_code += "\nclass supervisor supervisor;"

display(Markdown(f"```mermaid\n{mermaid_code}\n```"))

print_pool_state(pool, 'INITIAL STATE')

── Initial Architecture ─────────────────────────────────────


```mermaid
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	supervisor(supervisor)
	tool_manager(tool_manager)
	shell_agent(shell_agent)
	compute_agent(compute_agent)
	__end__([<p>__end__</p>]):::last
	__start__ --> supervisor;
	compute_agent --> supervisor;
	shell_agent --> supervisor;
	supervisor -.-> __end__;
	supervisor -.-> compute_agent;
	supervisor -.-> shell_agent;
	supervisor -.-> tool_manager;
	tool_manager --> supervisor;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

classDef supervisor fill:#f96,stroke:#333,stroke-width:4px;
class supervisor supervisor;
```


  INITIAL STATE
  Agents   : ['shell_agent', 'compute_agent']
  Registry : (empty)

  [shell_agent]
    base tools    : ['terminal']
    assigned tools: (none)
  [compute_agent]
    base tools    : (none)
    assigned tools: (none)


---
## 3 — Adding a Tool and a Dynamic Agent

We extend the system in two steps.

### Step A — Register a tool via the Python API

`boltzmann_energy` is registered directly through `pool.register_tool_from_code()`.
This is equivalent to asking the ToolManager via natural language but skips the LLM call.

**Rules for registrable functions:**
- Must have a **docstring** — this becomes the LLM's description of the tool
- Must be **top-level** (no nested classes)
- All imports must be **inside** the function body (the code is `exec`'d in an isolated namespace)
- Type annotations are recommended for argument validation by `StructuredTool`

### Step B — Create a dynamic agent via Supervisor prompt

We send a single natural language prompt asking the Supervisor to:
1. Create a new `thermal_agent` specialized in thermal calculations
2. Assign `boltzmann_energy` to it

The Supervisor routes both tasks to the ToolManager. When `add_agent_to_pool` is called,
the Supervisor is **automatically rebuilt** to include the new agent.

**Under the hood:**
```
pool.add_agent('thermal_agent', ...)
  ├── _rebuild_agent('thermal_agent')   # compile new ReAct agent
  ├── _rebuild_supervisor()             # recompile supervisor with 4 agents
  └── _autosave()                       # write pool_state.json
```

> **Persistence:** `thermal_agent` is created with `_is_dynamic=True` (the default
> when the ToolManager calls `add_agent_to_pool`). It is immediately written to
> `pool_state.json` and will be restored on the next startup.

In [7]:
# ── Step A: register boltzmann_energy directly via pool API ───────────────────
# Using single-quoted docstring inside a triple-single-quoted code block.

BOLTZMANN_CODE = '''
def boltzmann_energy(temperature_K: float) -> str:
    'Compute the Boltzmann thermal energy kT in eV for a given temperature in Kelvin. k_B = 8.617333e-5 eV/K.'
    kT = 8.617333e-5 * temperature_K
    return f'kT at {temperature_K:.1f} K = {kT:.6f} eV'
'''

result = pool.register_tool_from_code(BOLTZMANN_CODE)
print(f'Registration : {result}')
print(f'Registry now : {pool.list_registered_tools()}')

Registration : Registered: boltzmann_energy
Registry now : ['boltzmann_energy']


In [8]:
# ── Step B: create thermal_agent via a natural language Supervisor prompt ──────

config_setup = {'configurable': {'thread_id': 'setup'}}

setup_prompt = '''Please do the following steps in order:
1. Create a new agent called thermal_agent whose job is to compute thermal
   properties of materials using Boltzmann statistics.
2. Assign the boltzmann_energy tool to thermal_agent.
3. Confirm which agents are in the pool and which tools thermal_agent has.'''

print('Sending setup prompt to Supervisor...\n')
for chunk in pool.supervisor.stream(
    {'messages': [{'role': 'user', 'content': setup_prompt}]},
    config=config_setup,
    recursion_limit=20,
):
    pretty_print_messages(chunk, last_message=True)

Sending setup prompt to Supervisor...

Update from node supervisor:

================================= Tool Message =================================
Name: transfer_to_tool_manager

Successfully transferred to tool_manager

Update from node tool_manager:

================================= Tool Message =================================
Name: transfer_back_to_supervisor

Successfully transferred back to supervisor

Update from node supervisor:

================================== Ai Message ==================================
Name: supervisor

The new agent thermal_agent has been created and assigned the boltzmann_energy tool. The current agents in the pool are shell_agent, compute_agent, and thermal_agent. The thermal_agent has the boltzmann_energy tool assigned. How would you like to proceed?



### Updated Architecture

After adding `thermal_agent`, the Supervisor graph now has **four agent nodes**.
Notice that `pool.supervisor` reflects the new state automatically — no manual rebuild needed.

```
Supervisor
├── tool_manager   ← pool management
├── shell_agent    ← shell commands  [terminal]
├── compute_agent  ← general computation (still empty)
└── thermal_agent  ← thermal calculations  [boltzmann_energy]  ← NEW
```

In [1]:
print('── Updated Architecture (after adding thermal_agent) ─────────')
# display(Image(pool.supervisor.get_graph().draw_mermaid_png()))

mermaid_code = pool.supervisor.get_graph().draw_mermaid()
# Add a custom style for a node named 'supervisor'
mermaid_code += "\nclassDef supervisor fill:#f96,stroke:#333,stroke-width:4px;"
mermaid_code += "\nclass supervisor supervisor;"

display(Image(f"```mermaid\n{mermaid_code}\n```"))

print_pool_state(pool, 'STATE AFTER ADDING thermal_agent')

── Updated Architecture (after adding thermal_agent) ─────────


NameError: name 'pool' is not defined

---
## 4 — Adding a Second Tool from a File

`thermal_agent` currently has one tool: `boltzmann_energy`. We now extend it by
loading a second tool, `arrhenius_factor`, from a regular `.py` file on disk.

This demonstrates the **file-based tool registration** path, where functions are
defined in a normal Python module and loaded at runtime via the ToolManager.

### The Arrhenius equation

The rate of a thermally activated process follows the **Arrhenius equation**:

$$k(T) = A \cdot \exp\!\left(-\frac{E_a}{k_B T}\right)$$

where:
- **$E_a$** — activation energy barrier (eV)
- **$k_B T$** — Boltzmann thermal energy (eV)
- **$\exp(-E_a / k_B T)$** — the Boltzmann factor: fraction of particles with
  enough energy to overcome the barrier

The two tools are naturally **complementary**:

| Tool | Input | Output |
|------|-------|--------|
| `boltzmann_energy(T)` | temperature in K | kT in eV |
| `arrhenius_factor(Ea, kT)` | barrier (eV), kT (eV) | exp(−Ea/kT) |

> **Key:** The `arrhenius_factor` docstring explicitly instructs the LLM to first
> call `boltzmann_energy` to obtain `kT_eV`. This makes the tool chain
> self-documenting and guides the agent without extra prompting.

In [10]:
# Write the arrhenius_factor tool to a plain Python file
TOOLS_FILE = '/tmp/thermal_tools.py'

content = '''import math

def arrhenius_factor(activation_energy_eV: float, kT_eV: float) -> str:
    'Compute the Arrhenius Boltzmann factor exp(-Ea/kT). activation_energy_eV: barrier in eV. kT_eV: thermal energy in eV — use boltzmann_energy() to get this first.'
    factor = math.exp(-activation_energy_eV / kT_eV)
    return (f'Arrhenius factor: exp(-{activation_energy_eV:.4f} eV / {kT_eV:.6f} eV) = {factor:.6e}')
'''

with open(TOOLS_FILE, 'w') as f:
    f.write(content)

print(f'Written: {TOOLS_FILE}')
print()
print('File contents:')
print(content)

Written: /tmp/thermal_tools.py

File contents:
import math

def arrhenius_factor(activation_energy_eV: float, kT_eV: float) -> str:
    'Compute the Arrhenius Boltzmann factor exp(-Ea/kT). activation_energy_eV: barrier in eV. kT_eV: thermal energy in eV — use boltzmann_energy() to get this first.'
    factor = math.exp(-activation_energy_eV / kT_eV)
    return (f'Arrhenius factor: exp(-{activation_energy_eV:.4f} eV / {kT_eV:.6f} eV) = {factor:.6e}')



In [11]:
# Ask the Supervisor to load the file and assign the new tool to thermal_agent
config_file = {'configurable': {'thread_id': 'load-file'}}

file_prompt = (f'Load all tools from {TOOLS_FILE} '
               'and assign arrhenius_factor to thermal_agent. '
               'Then confirm the tools assigned to thermal_agent.')

print('Loading arrhenius_factor from file via Supervisor...\n')
for chunk in pool.supervisor.stream(
    {'messages': [{'role': 'user', 'content': file_prompt}]},
    config=config_file,
    recursion_limit=15,
):
    pretty_print_messages(chunk, last_message=True)

Loading arrhenius_factor from file via Supervisor...

Update from node supervisor:

================================= Tool Message =================================
Name: transfer_to_tool_manager

Successfully transferred to tool_manager

Update from node tool_manager:

================================= Tool Message =================================
Name: transfer_back_to_supervisor

Successfully transferred back to supervisor

Update from node supervisor:

================================== Ai Message ==================================
Name: supervisor

The tools from /tmp/thermal_tools.py have been loaded, and the tool arrhenius_factor has been assigned to thermal_agent. The thermal_agent currently has the tools boltzmann_energy and arrhenius_factor assigned.



In [12]:
print('── thermal_agent after loading from file ─────────────────────')
#display(Image(pool.supervisor.get_graph().draw_mermaid_png()))

mermaid_code = pool.supervisor.get_graph().draw_mermaid()
# Add a custom style for a node named 'supervisor'
mermaid_code += "\nclassDef supervisor fill:#f96,stroke:#333,stroke-width:4px;"
mermaid_code += "\nclass supervisor supervisor;"

display(Markdown(f"```mermaid\n{mermaid_code}\n```"))

print_pool_state(pool, 'STATE AFTER ADDING arrhenius_factor')

── thermal_agent after loading from file ─────────────────────


```mermaid
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	supervisor(supervisor)
	tool_manager(tool_manager)
	shell_agent(shell_agent)
	compute_agent(compute_agent)
	thermal_agent(thermal_agent)
	__end__([<p>__end__</p>]):::last
	__start__ --> supervisor;
	compute_agent --> supervisor;
	shell_agent --> supervisor;
	supervisor -.-> __end__;
	supervisor -.-> compute_agent;
	supervisor -.-> shell_agent;
	supervisor -.-> thermal_agent;
	supervisor -.-> tool_manager;
	thermal_agent --> supervisor;
	tool_manager --> supervisor;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

classDef supervisor fill:#f96,stroke:#333,stroke-width:4px;
class supervisor supervisor;
```


  STATE AFTER ADDING arrhenius_factor
  Agents   : ['shell_agent', 'compute_agent', 'thermal_agent']
  Registry : ['boltzmann_energy', 'arrhenius_factor']

  [shell_agent]
    base tools    : ['terminal']
    assigned tools: (none)
  [compute_agent]
    base tools    : (none)
    assigned tools: (none)
  [thermal_agent]
    base tools    : (none)
    assigned tools: ['boltzmann_energy', 'arrhenius_factor']


### Prompt Enhancer in Action — Two-Tool Chain

Now `thermal_agent` has **two assigned tools**: `boltzmann_energy` and `arrhenius_factor`.
We ask the question as a plain user would — no agent names, no tool names.

> *"A surface diffusion process on a metal catalyst has an activation barrier of
> 0.30 eV. What is the Arrhenius Boltzmann factor at 400 K?"*

The `PromptEnhancer` reads the current pool state and rewrites the query to include
explicit routing hints. The Supervisor receives the enhanced query and routes
directly to `thermal_agent`, which chains both tools:

```
Step 1 → boltzmann_energy(400)
         kT = 8.617333e-5 × 400 = 0.034469 eV

Step 2 → arrhenius_factor(0.30, 0.034469)
         exp(−0.30 / 0.034469) = exp(−8.703) ≈ 1.65 × 10⁻⁴
```

**Physical interpretation:** At 400 K, only ~0.016 % of surface atoms have
sufficient thermal energy to hop over a 0.30 eV diffusion barrier per attempt.

> **Without the enhancer**, the Supervisor receives the raw question and may require
> extra reasoning hops to discover which agent and tools to use. The enhancer
> eliminates that ambiguity by consulting the live pool before routing.

In [13]:
config_combined = {'configurable': {'thread_id': 'combined'}}

# ── Raw user query — no agent or tool names ────────────────────────────────────
raw_query = (
    'A surface diffusion process on a metal catalyst has an activation barrier of 0.30 eV. '
    'What is the Arrhenius Boltzmann factor at 400 K?'
)

print('User query (raw):')
print(f'  {raw_query}')
print()

# ── PromptEnhancer rewrites with routing hints ─────────────────────────────────
enhanced_query = enhancer.enhance(raw_query)

print('Enhanced query (sent to Supervisor):')
print(f'  {enhanced_query}')
print()
print('─' * 60)
print()

# ── Supervisor receives the enhanced query and routes to thermal_agent ─────────
for chunk in pool.supervisor.stream(
    {'messages': [{'role': 'user', 'content': enhanced_query}]},
    config=config_combined,
    recursion_limit=20,
):
    pretty_print_messages(chunk, last_message=True)

User query (raw):
  A surface diffusion process on a metal catalyst has an activation barrier of 0.30 eV. What is the Arrhenius Boltzmann factor at 400 K?

Enhanced query (sent to Supervisor):
  A surface diffusion process on a metal catalyst has an activation barrier of 0.30 eV. What is the Arrhenius Boltzmann factor at 400 K? Use thermal_agent — it should use arrhenius_factor and boltzmann_energy to compute the answer step by step.

────────────────────────────────────────────────────────────

Update from node supervisor:

================================= Tool Message =================================
Name: transfer_to_thermal_agent

Successfully transferred to thermal_agent

Update from node thermal_agent:

================================= Tool Message =================================
Name: transfer_back_to_supervisor

Successfully transferred back to supervisor

Update from node supervisor:

================================= Tool Message =================================
Name: t

---
## 5 — Removing the Agent and Tool

DynaMate supports runtime removal of dynamic agents and tools through natural language
prompts. The ToolManager exposes two removal tools:

| ToolManager tool | Pool method called | Side effects |
|-----------------|-------------------|-------------|
| `remove_agent_from_pool(name)` | `pool.remove_agent(name)` | Removes agent; rebuilds supervisor |
| `remove_tool_from_registry(name)` | `pool.remove_tool(name)` | Unregisters tool; unassigns from all agents; rebuilds affected agents |

**Protected agents** — `shell_agent`, `compute_agent`, and `tool_manager` cannot
be removed through the ToolManager (they are always rebuilt fresh on startup).

**Persistence** — both operations immediately update `pool_state.json`.
The removed agent and tool will not appear in the next session.

> **Tip:** You can also call `pool.remove_agent()` and `pool.remove_tool()` directly
> via the Python API, bypassing the LLM entirely.

In [14]:
config_cleanup = {'configurable': {'thread_id': 'cleanup'}}

cleanup_prompt = '''Please do the following steps in order:
1. Remove thermal_agent from the pool.
2. Remove boltzmann_energy from the tool registry.
3. Confirm the current agents and tool registry to verify the cleanup.'''

print('Sending cleanup prompt to Supervisor...\n')
for chunk in pool.supervisor.stream(
    {'messages': [{'role': 'user', 'content': cleanup_prompt}]},
    config=config_cleanup,
    recursion_limit=15,
):
    pretty_print_messages(chunk, last_message=True)

Sending cleanup prompt to Supervisor...

Update from node supervisor:

================================= Tool Message =================================
Name: transfer_to_tool_manager

Successfully transferred to tool_manager

Update from node tool_manager:

================================= Tool Message =================================
Name: transfer_back_to_supervisor

Successfully transferred back to supervisor

Update from node supervisor:

================================== Ai Message ==================================
Name: supervisor

The removal of thermal_agent and boltzmann_energy has been completed. The current agents are shell_agent and compute_agent. The only tool in the registry now is arrhenius_factor. How would you like to proceed?



### Restored Architecture

After removing `thermal_agent` and `boltzmann_energy`, the pool returns to its
original three-agent configuration:

```
Supervisor
├── tool_manager   ← pool management
├── shell_agent    ← shell commands  [terminal]
└── compute_agent  ← no tools (same as initial state)
```

The `pool_state.json` file is now empty (`tools: {}`, `dynamic_agents: {}`,
`assignments: {}`), so the next session starts clean.

In [15]:
print('── Restored Architecture (after removing thermal_agent) ──────')
#display(Image(pool.supervisor.get_graph().draw_mermaid_png()))

mermaid_code = pool.supervisor.get_graph().draw_mermaid()
# Add a custom style for a node named 'supervisor'
mermaid_code += "\nclassDef supervisor fill:#f96,stroke:#333,stroke-width:4px;"
mermaid_code += "\nclass supervisor supervisor;"

display(Markdown(f"```mermaid\n{mermaid_code}\n```"))

print_pool_state(pool, 'RESTORED STATE')

# Verify the JSON store is clean
import json
with open(os.path.join(TMP, 'pool_state.json')) as f:
    saved = json.load(f)
print()
print('pool_state.json contents:')
print(json.dumps(saved, indent=2))

── Restored Architecture (after removing thermal_agent) ──────


```mermaid
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	supervisor(supervisor)
	tool_manager(tool_manager)
	shell_agent(shell_agent)
	compute_agent(compute_agent)
	__end__([<p>__end__</p>]):::last
	__start__ --> supervisor;
	compute_agent --> supervisor;
	shell_agent --> supervisor;
	supervisor -.-> __end__;
	supervisor -.-> compute_agent;
	supervisor -.-> shell_agent;
	supervisor -.-> tool_manager;
	tool_manager --> supervisor;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

classDef supervisor fill:#f96,stroke:#333,stroke-width:4px;
class supervisor supervisor;
```


  RESTORED STATE
  Agents   : ['shell_agent', 'compute_agent']
  Registry : ['arrhenius_factor']

  [shell_agent]
    base tools    : ['terminal']
    assigned tools: (none)
  [compute_agent]
    base tools    : (none)
    assigned tools: (none)

pool_state.json contents:
{
  "tools": {
    "arrhenius_factor": "import math\n\ndef arrhenius_factor(activation_energy_eV: float, kT_eV: float) -> str:\n    'Compute the Arrhenius Boltzmann factor exp(-Ea/kT). activation_energy_eV: barrier in eV. kT_eV: thermal energy in eV \u2014 use boltzmann_energy() to get this first.'\n    factor = math.exp(-activation_energy_eV / kT_eV)\n    return (f'Arrhenius factor: exp(-{activation_energy_eV:.4f} eV / {kT_eV:.6f} eV) = {factor:.6e}')\n"
  },
  "dynamic_agents": {},
  "assignments": {}
}


---
## Summary

This tutorial demonstrated the complete add–use–remove lifecycle of DynaMate,
with the **Prompt Enhancer** bridging the gap between plain user language and
the supervisor's routing logic:

| Phase | Action | Internal effect |
|-------|--------|-----------------|
| Init | Build pool + supervisor + enhancer | 3-agent supervisor compiled; PromptEnhancer bound to live pool |
| Add | Register `boltzmann_energy` | Stored in `_tool_registry` + `_source_registry`; JSON updated |
| Add | Create `thermal_agent` (via Supervisor prompt) | 4-agent supervisor rebuilt; JSON updated |
| Add | Load and assign `arrhenius_factor` (from file) | `thermal_agent` rebuilt with 2 tools; JSON updated |
| **Use** | **User asks raw question → enhancer rewrites → supervisor routes** | **PromptEnhancer inspects pool, appends routing hint; thermal_agent chains both tools** |
| Remove | Remove `thermal_agent` (via Supervisor prompt) | 3-agent supervisor rebuilt; JSON updated |
| Remove | Remove `boltzmann_energy` (via Supervisor prompt) | Registry entry deleted; no agent rebuilds needed |

### Key properties demonstrated

- **Prompt Enhancer** — translates natural user language into explicit agent+tool routing hints; context is always live (no restarts needed after pool changes)
- **Targeted rebuilds** — only the affected agent or supervisor is recompiled on each change
- **Natural language interface** — add and remove agents/tools via plain English prompts
- **Automatic persistence** — every mutation is written to `pool_state.json` synchronously
- **Introspectable state** — `pool.supervisor.get_graph().draw_mermaid_png()` always reflects the live compiled graph
- **Protected agents** — initial agents (`shell_agent`, `compute_agent`, `tool_manager`) cannot be removed through the ToolManager

### Going further

```bash
# Start an interactive persistent session (state saved across restarts)
python main.py

# Continue a named conversation thread
python main.py --thread-id my-project

# Inspect graph and pool state without starting a session
python main.py --status
python misc/show_graph.py --png graph.png
```

**Other things to try:**
- Register tools from a `.py` file: `pool.register_tool_from_file('/path/to/tools.py')`
- Assign the same tool to multiple agents — each is rebuilt independently
- Create multiple specialized agents (e.g. `unit_conversion_agent`, `statistics_agent`)
- Inspect `pool_state.json` directly — it is human-readable JSON
- Query conversation history using any SQLite browser on `.dynamate/conversations`